In [0]:
CREATE OR REPLACE TABLE workspace.retail_project.gold_sales_summary 
COMMENT 'Gold sales summary. Rows = Silver rows where Quantity > 0 AND is_cancellation = FALSE. Cancellation and other negative-quantity rows remain in Ailver for seperate reporting. Sales value delibarately not computed here'
AS 
SELECT
    InvoiceNo,
    StockCode,
    Description,
    Quantity,
    InvoiceDate,
    UnitPrice,
    CustomerID,
    Country
FROM workspace.retail_project.silver_online_retail
WHERE Quantity > 0 AND is_cancellation = FALSE;

num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*) 
FROM workspace.retail_project.gold_sales_summary;

count(1)
531285


In [0]:
SELECT 
  COUNT(*) AS gold_rows,
  COUNT(DISTINCT InvoiceNo) AS distinct_invoices,
  MIN(InvoiceDate) AS earliest_invoice_date,
  MAX(InvoiceDate) AS latest_invoice_date,
  COUNT(DISTINCT DATE(InvoiceDate)) AS distinct_invoice_dates
FROM 
  workspace.retail_project.gold_sales_summary;

gold_rows,distinct_invoices,earliest_invoice_date,latest_invoice_date,distinct_invoice_dates
531285,20728,2010-12-01T08:26:00.000Z,2011-12-09T12:50:00.000Z,305


In [0]:
CREATE OR REPLACE TABLE workspace.retail_project.gold_sales_daily 
COMMENT 'Daily Gold sales summary. One row per calendar date derived from gold_sales_summary (positive Quantity AND is_cancellation = FALSE). Revenue = SUM(Quantity * UnitPrice) in source currency, computed per line then summed. Excludes cancellation and other negative-quantity rows, which remain in Silver for separate reporting'
AS 
SELECT
  DATE(InvoiceDate) AS sales_date,
  COUNT(*) AS sales_line_count,
  COUNT(DISTINCT InvoiceNo) AS distinct_invoice_count,
  SUM(Quantity) AS units_sold,
  CAST(SUM(Quantity * UnitPrice) AS DECIMAL(18, 4)) AS revenue
FROM workspace.retail_project.gold_sales_summary
WHERE InvoiceDate IS NOT NULL
GROUP BY DATE(InvoiceDate)
ORDER BY sales_date;


num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*),
SUM(revenue)
FROM workspace.retail_project.gold_sales_daily;

count(1),sum(revenue)
305,10644560.4240


In [0]:
SELECT
    (SELECT SUM(units_sold) FROM workspace.retail_project.gold_sales_daily)        AS daily_units,
    (SELECT SUM(Quantity)   FROM workspace.retail_project.gold_sales_summary)      AS line_units,
    (SELECT SUM(units_sold) FROM workspace.retail_project.gold_sales_daily)
      - (SELECT SUM(Quantity) FROM workspace.retail_project.gold_sales_summary)    AS units_diff,

    (SELECT SUM(revenue) FROM workspace.retail_project.gold_sales_daily)           AS daily_revenue,
    (SELECT SUM(Quantity * UnitPrice) FROM workspace.retail_project.gold_sales_summary) AS line_revenue,
    (SELECT SUM(revenue) FROM workspace.retail_project.gold_sales_daily)
      - (SELECT SUM(Quantity * UnitPrice) FROM workspace.retail_project.gold_sales_summary) AS revenue_diff,

    (SELECT SUM(sales_line_count) FROM workspace.retail_project.gold_sales_daily)  AS daily_line_count,
    (SELECT COUNT(*)              FROM workspace.retail_project.gold_sales_summary) AS line_count,
    (SELECT SUM(sales_line_count) FROM workspace.retail_project.gold_sales_daily)
      - (SELECT COUNT(*) FROM workspace.retail_project.gold_sales_summary)         AS lines_diff

;

daily_units,line_units,units_diff,daily_revenue,line_revenue,revenue_diff,daily_line_count,line_count,lines_diff
5660981,5660981,0,10644560.4240,10644560.4240,0.0000,531285,531285,0


In [0]:
CREATE OR REPLACE TABLE workspace.retail_project.gold_sales_monthly
COMMENT 'Monthly Gold sales trend. One row per calendar month, built directly from gold_sales_summary (line-level) so distinct invoice counts are exact. Revenue = SUM(Quantity * UnitPrice), computed per line then summed, in source currency. Excludes cancellation and other negative-quantity rows, which remain in Silver.'
AS
SELECT
    DATE_TRUNC('MONTH', InvoiceDate) AS sales_month,
    COUNT(*) AS sales_line_count,
    COUNT(DISTINCT InvoiceNo) AS distinct_invoice_count,
    SUM(Quantity) AS units_sold,
    CAST(SUM(Quantity * UnitPrice) AS DECIMAL(18,4)) AS revenue
FROM workspace.retail_project.gold_sales_summary
WHERE InvoiceDate IS NOT NULL
GROUP BY DATE_TRUNC('MONTH', InvoiceDate)
ORDER BY sales_month;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE workspace.retail_project.gold_sales_by_country
COMMENT 'Gold sales by country. One row per Country, built directly from gold_sales_summary. Revenue = SUM(Quantity * UnitPrice)'
AS
SELECT
    Country AS country,
    COUNT(*) AS sales_line_count,
    COUNT(DISTINCT InvoiceNo) AS distinct_invoice_count,
    SUM(Quantity) AS units_sold,
    CAST(SUM(Quantity * UnitPrice) AS DECIMAL(18,4)) AS revenue
FROM workspace.retail_project.gold_sales_summary
GROUP BY Country
ORDER BY revenue DESC;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE workspace.retail_project.gold_sales_by_stockcode
COMMENT 'Gold sales by StockCode. One row per StockCode, built directly from gold_sales_summary. Grouped by StockCode only, NOT Description, so a code with multiple descriptions stays one row'
AS
SELECT
    StockCode AS stock_code,
    COUNT(*) AS sales_line_count,
    COUNT(DISTINCT InvoiceNo) AS distinct_invoice_count,
    SUM(Quantity) AS units_sold,
    CAST(SUM(Quantity * UnitPrice) AS DECIMAL(18,4)) AS revenue
FROM workspace.retail_project.gold_sales_summary
GROUP BY StockCode;

num_affected_rows,num_inserted_rows


In [0]:
SELECT
    -- Grain check: one row per distinct StockCode
    (SELECT COUNT(*)                                FROM workspace.retail_project.gold_sales_by_stockcode) AS stockcode_rows,
    (SELECT COUNT(DISTINCT stock_code)              FROM workspace.retail_project.gold_sales_by_stockcode) AS distinct_stockcodes,
    (SELECT COUNT(*)                                FROM workspace.retail_project.gold_sales_by_stockcode)
      - (SELECT COUNT(DISTINCT stock_code)          FROM workspace.retail_project.gold_sales_by_stockcode) AS grain_diff,

    -- Units: stockcode rollup vs. line-level
    (SELECT SUM(units_sold)                         FROM workspace.retail_project.gold_sales_by_stockcode) AS sc_units,
    (SELECT SUM(Quantity)                           FROM workspace.retail_project.gold_sales_summary)      AS line_units,
    (SELECT SUM(units_sold)                         FROM workspace.retail_project.gold_sales_by_stockcode)
      - (SELECT SUM(Quantity)                       FROM workspace.retail_project.gold_sales_summary)      AS units_diff,

    -- Revenue: stockcode rollup vs. line-level
    (SELECT SUM(revenue)                            FROM workspace.retail_project.gold_sales_by_stockcode) AS sc_revenue,
    (SELECT SUM(Quantity * UnitPrice)               FROM workspace.retail_project.gold_sales_summary)      AS line_revenue,
    (SELECT SUM(revenue)                            FROM workspace.retail_project.gold_sales_by_stockcode)
      - (SELECT SUM(Quantity * UnitPrice)           FROM workspace.retail_project.gold_sales_summary)      AS revenue_diff
;

stockcode_rows,distinct_stockcodes,grain_diff,sc_units,line_units,units_diff,sc_revenue,line_revenue,revenue_diff
3941,3941,0,5660981,5660981,0,10644560.4240,10644560.4240,0.0000
